In [ ]:
!pip install openai
!pip install pymed

In [ ]:
!pip install setuptools==57.5.0
!pip install scholarly

In [ ]:
from IPython.display import display, Markdown
import re
import pandas as pd
import itertools
from tqdm import tqdm

from openai import OpenAI
from pymed import PubMed
from scholarly import scholarly

## OpenAI

In [ ]:
client = OpenAI(
    # This is the default and can be omitted
    api_key='YOUR_API_KEY',
)

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    instructions=(
        "You are a systematic review assistant. Your goal is to replicate a human literature search "
        "using the same queries and logic. Use the search terms and filters from a provided human review, "
        "simulate retrieval from databases such as PubMed, Web of Science, Scopus, and EBSCOhost, "
        "and return a list of titles, abstracts, and metadata for potentially relevant articles. "
        "Do not invent articles. If information is missing or unclear, explain what assumptions you made."
        "Include journal names and authors"
    ),
    input="Run a federated literature search for long COVID symptom clusters using the following search terms:\n"
          "- Keywords: long covid OR chronic covid OR post covid syndrome OR covid OR covid-19 OR long haul OR post-acute covid\n"
          "- Symptoms: fatigue, anxiety, depression, cognitive impairment, shortness of breath, chest pain\n"
          "- Databases: PubMed, Scopus, Web of Science, EBSCOhost\n"
          "- Filters: publication years 2020-2022, English only"
)

print(response.output_text)

In [ ]:
result_number_regex = re.compile(r'^\d+\.')
data_title_regex = re.compile(r'\*\*(Title|Authors|Journal|Year|Abstract):\*\*')

rows = []
row = {}
for line in response.output_text.splitlines():
    if result_number_regex.search(line):
        if row:
            rows.append(row)
        row = {}
    match = data_title_regex.search(line)
    if not match:
        continue
    row[match.group(1)] = line[match.end():].strip()
if row:
    rows.append(row)
df = pd.DataFrame(rows)
df

In [ ]:
df.to_csv('openai_results.csv', index=False)

In [ ]:
Markdown(response.output_text)

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "developer", "content": "Talk like a pirate."},
        {
            "role": "user",
            "content": "How do I check if a Python object is an instance of a class?",
        },
    ],
)

Markdown(completion.choices[0].message.content)

For more, see https://github.com/openai/openai-python

## PubMed

In [ ]:
# Use work address instead
pubmed = PubMed(tool='BioC', email="you@example.org")

In [ ]:
results = pubmed.query('((((("long covid" OR "chronic covid" OR "post covid syndrome" OR covid OR covid-19 OR "long haul" OR "post-acute covid")) AND ((cough OR pain OR "shortness of breath" OR dyspn* OR palpitations OR heart failure OR "heart attack" OR "transient ischemia" OR "myocardial infarction" OR arrythmia OR stroke OR "cardiovascular accident" OR DVT OR "thromb*" OR PE OR "pulmonary embo*"))) AND ((fatigue OR weakness OR "joint pain" OR arthralgia OR "loss of smell" OR anosmia OR ageusia OR "loss of taste" OR seizure OR "visual problems" OR diplopia OR "peripheral neuropathy" OR "loss of sensation" OR paresthesia OR tinnitus OR diarrhea OR "loss of appetite" OR "muscle pain" OR myalgia ))) AND ((constipation OR nausea OR vomiting OR " abdominal pain" OR insomnia OR depression OR anxiety OR "decreased cognitive function"))) AND ((constipation OR nausea OR vomiting OR " abdominal pain" OR insomnia OR depression OR anxiety OR "decreased cognitive function")) AND ("2021/12/28"[Date - Publication] : "2022/12/28"[Date - Publication]) ', max_results=600)
#results = pubmed.query("""("Asthma"[Mesh] OR "Dermatitis, Atopic"[Mesh] OR "Food Hypersensitivity"[Mesh] OR "Rhinitis, Allergic"[Mesh]  OR asthma[Title/Abstract] OR "atopic dermatitis"[Title/Abstract] OR eczema[Title/Abstract]   OR "food allergy"[Title/Abstract] OR "food hypersensitivity"[Title/Abstract]   OR "allergic rhinitis"[Title/Abstract] OR "hay fever"[Title/Abstract] OR aeroallergen*[Title/Abstract] ) AND (  "Prevalence"[Mesh] OR "Epidemiology"[Subheading]   OR prevalence[Title/Abstract] OR epidemiology[Title/Abstract] OR "cross-sectional"[Title/Abstract]   OR "population-based"[Title/Abstract] OR "health survey"[Title/Abstract] OR "morbidity rate"[Title/Abstract] ) AND ( "Israel"[Mesh] OR Israel[Title/Abstract] OR "Israeli population"[Title/Abstract] OR "Jewish population"[Title/Abstract] OR "Arab population"[Title/Abstract]""")
#results = pubmed.query("""(("Asthma"[Majr] OR "Dermatitis, Atopic"[Majr] OR "Food Hypersensitivity"[Majr] OR "Rhinitis, Allergic"[Majr])  OR (asthma[Title] OR "atopic dermatitis"[Title] OR eczema[Title] OR "food allergy"[Title] OR "allergic rhinitis"[Title])) AND (prevalence[Title/Abstract] OR epidemiology[Title/Abstract] OR "time trend"[Title/Abstract] OR longitudinal[Title/Abstract]) AND (Israel[Title/Abstract] OR "Israeli population"[Title/Abstract])""", max_results=600)
#results = pubmed.query("""("Asthma"[Mesh] OR "Dermatitis, Atopic"[Mesh] OR "Food Hypersensitivity"[Mesh] OR "Rhinitis, Allergic"[Mesh]  OR asthma[Title/Abstract] OR "atopic dermatitis"[Title/Abstract] OR eczema[Title/Abstract] OR "food allergy"[Title/Abstract]  OR "food hypersensitivity"[Title/Abstract] OR "allergic rhinitis"[Title/Abstract] OR "hay fever"[Title/Abstract] OR aeroallergen*[Title/Abstract])  AND  ("Prevalence"[Mesh] OR "Epidemiology"[Subheading] OR prevalence[Title/Abstract] OR epidemiology[Title/Abstract]  OR "cross-sectional"[Title/Abstract] OR "population-based"[Title/Abstract] OR "health survey"[Title/Abstract]  OR "morbidity rate"[Title/Abstract] OR "trend*"[Title/Abstract] OR "temporal"[Title/Abstract]  OR "longitudinal"[Title/Abstract] OR "time series"[Title/Abstract] OR "observational"[Title/Abstract] OR "registry"[Title/Abstract])  AND  ("Israel"[Mesh] OR Israel[Title/Abstract] OR "Israeli population"[Title/Abstract]  OR "Jewish population"[Title/Abstract] OR "Arab population"[Title/Abstract])""", max_results=600)
#results = pubmed.query("""(((enabler*[Title/Abstract] OR barrier*[Title/Abstract] OR facilitator*[Title/Abstract] OR obstacle[Title/Abstract] OR challenge[Title/Abstract]) AND (vaccin*[Title/Abstract] OR immuniz*[Title/Abstract] OR immunis*[Title/Abstract] OR innoculation[Title/Abstract] OR "human papillomavirus vaccines"[MeSH Terms] OR HPV[Title/Abstract] OR "measles mumps rubella"[Title/Abstract] OR MMR[Title/Abstract]) AND (provider[Title/Abstract] OR "access to care"[Title/Abstract] OR insurance[Title/Abstract] OR "cultural competence"[Title/Abstract] OR finance[Title/Abstract] OR coverage[Title/Abstract]) AND (underserved[Title/Abstract] OR disadvantaged[Title/Abstract] OR minority[Title/Abstract] OR "hard to reach"[Title/Abstract] OR "difficult to reach"[Title/Abstract] OR vulnerable[Title/Abstract] OR "religious group"[Title/Abstract] OR "ethnic group"[Title/Abstract] OR migrant[Title/Abstract] OR nomad[Title/Abstract] OR refugee*[Title/Abstract] OR "asylum seek*"[Title/Abstract] OR immigrant[Title/Abstract] OR marginalized[Title/Abstract] OR periphery[Title/Abstract] OR rural[Title/Abstract] OR discriminated[Title/Abstract] OR deprived[Title/Abstract]))
#NOT
#(COVID[Title/Abstract] OR CORONA[Title/Abstract])
#AND
#(English[lang])
#AND
#("2010/01/01"[Date - Publication] : "3000"[Date - Publication])""", max_results=600)

#results = pubmed.query("""( "Asthma"[Mesh] OR "Dermatitis, Atopic"[Mesh] OR "Food Hypersensitivity"[Mesh] OR "Rhinitis, Allergic"[Mesh] OR asthma[Title/Abstract] OR "atopic dermatitis"[Title/Abstract] OR eczema[Title/Abstract] OR "food allergy"[Title/Abstract] OR "food hypersensitivity"[Title/Abstract] OR "allergic rhinitis"[Title/Abstract] OR "hay fever"[Title/Abstract] OR aeroallergen[Title/Abstract] OR atopy[Title/Abstract] OR atopic[Title/Abstract] OR "allergic diseases"[Title/Abstract] OR "inhalant allergy"[Title/Abstract]) AND ("Prevalence"[Mesh] OR "Epidemiology"[Subheading] OR "Incidence"[Mesh] OR prevalence[Title/Abstract] OR incidence[Title/Abstract] OR epidemiology[Title/Abstract] OR "cross-sectional"[Title/Abstract] OR "population-based"[Title/Abstract] OR "health survey"[Title/Abstract] OR "morbidity rate"[Title/Abstract] OR rate[Title/Abstract] OR frequency[Title/Abstract] OR distribution[Title/Abstract] OR trend[Title/Abstract] OR trends[Title/Abstract]) AND ("Israel"[Mesh] OR Israel[Title/Abstract] OR "Israeli population"[Title/Abstract] OR "Jewish population"[Title/Abstract] OR "Arab population"[Title/Abstract] OR "Israeli children"[Title/Abstract] OR "Israeli adults"[Title/Abstract])""", max_results=600)

"""( "Asthma"[Mesh] OR "Dermatitis, Atopic"[Mesh] OR "Food Hypersensitivity"[Mesh] OR "Rhinitis, Allergic"[Mesh] OR asthma[Title/Abstract] OR "atopic dermatitis"[Title/Abstract]
OR eczema[Title/Abstract] OR "food allergy"[Title/Abstract] OR "food hypersensitivity"[Title/Abstract] OR "allergic rhinitis"[Title/Abstract] OR "hay fever"[Title/Abstract] OR
aeroallergen*[Title/Abstract] OR atopy[Title/Abstract] OR atopic[Title/Abstract] OR "allergic diseases"[Title/Abstract] OR "inhalant allergy"[Title/Abstract]) AND
("Prevalence"[Mesh] OR "Epidemiology"[Subheading] OR "Incidence"[Mesh] OR prevalence[Title/Abstract] OR incidence[Title/Abstract] OR epidemiology[Title/Abstract] OR
"cross-sectional"[Title/Abstract] OR "population-based"[Title/Abstract] OR "health survey"[Title/Abstract] OR "morbidity rate"[Title/Abstract] OR rate[Title/Abstract] OR
frequency[Title/Abstract] OR distribution[Title/Abstract] OR trend[Title/Abstract] OR trends[Title/Abstract]) AND ("Israel"[Mesh] OR Israel[Title/Abstract] OR "Israeli
population"[Title/Abstract] OR "Jewish population"[Title/Abstract] OR "Arab population"[Title/Abstract] OR "Israeli children"[Title/Abstract] OR "Israeli adults"[Title/Abstract])
"""

#results = pubmed.query("""(
#  Enabler*[tiab] OR Barrier*[tiab] OR Facilitator*[tiab] OR Obstacle[tiab] OR Challenge[tiab]
#)
#AND
##(
##  Vaccin* OR Immuniz* OR Immunis* OR Innoculation OR 
#  "Human papillomavirus vaccines" OR HPV OR 
#  "Measles mumps rubella" OR MMR
#)
#AND
#(
#  Provider OR "Access to care" OR Insurance OR 
#  "Cultural competence" OR Finance OR Coverage OR 
#  "Service delivery" OR "Health Workforce" OR Leadership OR Governance
#)
#AND
#(
#  Underserved OR Disadvantaged OR Minority OR 
#  "Hard to reach" OR "Difficult to reach" OR Vulnerable OR 
#  "Religious group" OR "Ethnic group" OR Migrant OR Nomad OR Refugee* OR 
#  "Asylum seek*" OR Immigrant OR Marginalized OR Periphery OR Rural OR 
#  Discriminated OR Deprived
#)
#NOT
#(
#  COVID OR CORONA
#)
#AND ("2010/01/01"[Date - Publication] : "2021/12/31"[Date - Publication])
#AND (English[Language])




#""", max_results=1000)
results = list(results)
len(results)

In [ ]:
rows = []
for result in results:
    row = {}
    for field in ['title', 'publication_date', 'abstract']:
        row[field] = getattr(result, field)
    row['authors'] = ', '.join(
        f'{author["lastname"]}, {author["initials"]}.' for author in result.authors)
    row['pubmed_id'] = result.pubmed_id.splitlines()[0]
    rows.append(row)
df = pd.DataFrame(rows)
df

In [ ]:
#df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\pubmed_results_224.xlsx', index=False)
df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\pubmed_results_549.xlsx', index=False)

In [ ]:
alt_df = pd.read_excel(r'..\covid\549_results_from_PUBMED.xlsx')

In [ ]:
alt_df.columns

In [ ]:
set(alt_df.PMID) == set(df.pubmed_id.astype(int))

In [ ]:
len(alt_df), len(df)

In [ ]:
df.pubmed_id.dtype, alt_df.PMID.dtype

In [ ]:
results[2].title

In [ ]:
results[2].authors

In [ ]:
results[2].pubmed_id

Fields:

* abstract
* authors
* conclusions
* copyrights
* doi
* journal
* keywords
* methods
* publication_date
* pubmed_id
* results
* title

## Scopus

In [ ]:
import pybliometrics
from pybliometrics.scopus import AbstractRetrieval

In [ ]:
pybliometrics.init()

## Scholarly

For docs, see https://pypi.org/project/scholarly/

IMPORTANT: Making certain types of queries, such as scholarly.citedby or scholarly.search_pubs, will lead to Google Scholar blocking your requests and may eventually block your IP address. You must use proxy services to avoid this situation. 

In [ ]:
from scholarly import scholarly
from scholarly import ProxyGenerator

In [ ]:
print('\n'.join(f'scholarly.{method}' for method in dir(scholarly) if method.startswith('search')))

In [ ]:
pg = ProxyGenerator()
pg.FreeProxies()
scholarly.use_proxy(pg, pg)

In [ ]:
pubs = scholarly.search_pubs('long covid articles consequences OR impacts OR symptoms "long covid" OR "chronic covid"')

In [ ]:
MAX_RESULTS = 10

In [ ]:
rows = []
for result in tqdm(itertools.islice(pubs, MAX_RESULTS)):
    bib = result['bib']
    rows.append({
        'title': bib['title'],
        'journal': bib['venue'],
        'authors': ', '.join(bib['author'])
    })

df = pd.DataFrame(rows)
df

In [ ]:
df.to_excel('scholar.xlsx')

In [ ]:
results = [bigger]
for i in range(10):
    result = next(pubs)
    bib = result['bib']
    print(f"{i+1}. {bib['title']}")
    result = scholarly.fill(result)
    results.append(result)
    print(bib['title'])
    print(bib['author'])
    print(bib['journal'])


## resp

In [ ]:
from resp.apis.serp_api import Serp
from resp.apis.cnnp import connected_papers
from resp.apis.semantic_s import Semantic_Scholar
from resp.apis.acm_api import ACM
from resp.apis.arxiv_api import Arxiv
from resp.resp import Resp

In [ ]:
import resp
dir(resp)

In [ ]:
resp.__doc__